# Conditional WGAN-GP on CelebA, Conditioned on Blond Hair

Adapted from Foster, D. (2023). *Generative Deep Learning* (2nd ed.), notebook `04_gan/03_cgan/cgan.ipynb`
(https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition), itself adapted from Sayak Paul's Keras CGAN tutorial.

**Assignment tasks**
1. Condition the CGAN on the **`Blond_Hair`** attribute of CelebA (label 0 = not blond, label 1 = blond).
2. Change `train_step` so the labels match the input format of each network: the **generator** takes a one-hot *vector* `(batch, 2)`; the **critic** takes a one-hot *map* `(batch, 64, 64, 2)` stacked onto the image channels.

**Other changes for current Google Colab (Keras 3)**
* CelebA is downloaded from Kaggle with `kagglehub` instead of the book's Docker path `/app/data/...`
* Attribute rows are sorted by file name so every label lines up with its image
* The generator and critic are checkpointed separately with `.weights.h5` files (Keras 3 cannot checkpoint the unbuilt wrapper model), and saved models use `.keras`; the book's `notebooks.utils` helpers are replaced by a local `display()`
* `run_eagerly=True` removed so training runs as a compiled graph (much faster)
* Images are shown every 200 steps instead of every step, and loss is logged every 100 steps
* A conditioning-accuracy check at the end: a separate blond-hair classifier, trained on real faces, judges the generated faces

**Before running:** Runtime > Change runtime type > **T4 GPU**. Training takes roughly 15 to 30 minutes.

In [ ]:
# 1. Imports
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, utils, metrics, optimizers

print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. Parameters (same as the textbook, plus a few display settings)
IMAGE_SIZE = 64
CHANNELS = 3
CLASSES = 2
BATCH_SIZE = 128
Z_DIM = 32
LEARNING_RATE = 0.00005
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.9
EPOCHS = 20            # the book trains for EPOCHS * 100 single-batch steps = 2,000 steps
CRITIC_STEPS = 3
GP_WEIGHT = 10.0
LOAD_MODEL = False
LABEL = 'Blond_Hair'   # TASK 1: condition on the blond hair attribute

SHOW_EVERY = 200       # show generated faces every N steps
LOG_EVERY = 100        # print losses every N steps
TRAIN_STEPS = int(os.environ.get('TRAIN_STEPS', EPOCHS * 100))

os.makedirs('./output', exist_ok=True)
os.makedirs('./checkpoint', exist_ok=True)
os.makedirs('./models', exist_ok=True)
tf.random.set_seed(42); np.random.seed(42)

## 1. Prepare the data
CelebA on Kaggle: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset (about 1.3 GB, 202,599 faces with 40 attributes).

In [ ]:
# Download CelebA from Kaggle (Colab usually serves it from a fast cache)
DATA_DIR = os.environ.get('CELEBA_DIR')   # optional override
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download('jessicali9530/celeba-dataset')

ATTR_FILE = glob.glob(os.path.join(DATA_DIR, '**', 'list_attr_celeba.csv'), recursive=True)[0]
first_img = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.jpg'), recursive=True))[0]
IMG_DIR = os.path.dirname(first_img)
print('Attributes:', ATTR_FILE)
print('Images:    ', IMG_DIR)

In [ ]:
# Load the attribute table and look at the Blond_Hair column
attributes = pd.read_csv(ATTR_FILE)
attributes = attributes.sort_values('image_id').reset_index(drop=True)   # match the file order used below
print(attributes.columns.tolist())

n_blond = int((attributes[LABEL] == 1).sum())
print(f'{len(attributes):,} faces; {n_blond:,} blond ({n_blond / len(attributes):.1%}), '
      f'{len(attributes) - n_blond:,} not blond')
attributes[['image_id', LABEL]].head()

In [ ]:
# Labels: CelebA uses +1 / -1, convert to 1 (blond) / 0 (not blond)
int_labels = [1 if x == 1 else 0 for x in attributes[LABEL]]

train_data = utils.image_dataset_from_directory(
    IMG_DIR,
    labels=int_labels,
    color_mode='rgb',
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    interpolation='bilinear',
)

def preprocess(img):
    """Scale pixels from [0, 255] to [-1, 1] to match the generator's tanh output."""
    return (tf.cast(img, 'float32') - 127.5) / 127.5

train = train_data.map(
    lambda x, y: (preprocess(x), tf.one_hot(y, depth=CLASSES)),
    num_parallel_calls=tf.data.AUTOTUNE,
).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Display helper (replaces the book's notebooks.utils.display)
def display(images, n=10, title=None, save_to=None, labels=None):
    images = np.asarray(images)
    if images.min() < 0:                       # [-1, 1] -> [0, 1]
        images = (images + 1) / 2
    elif images.max() > 1:                     # [0, 255] -> [0, 1]
        images = images / 255.0
    n = min(n, len(images))
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.3))
    for i, ax in enumerate(np.atleast_1d(axes)):
        ax.imshow(np.clip(images[i], 0, 1))
        ax.axis('off')
        if labels is not None:
            ax.set_title(labels[i], fontsize=9)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    if save_to:
        plt.savefig(save_to, dpi=100)
    plt.show()

# Show some real faces with their Blond_Hair label
x_batch, y_batch = next(iter(train))
display(x_batch[:10], labels=['blond' if y[1] == 1 else 'not blond' for y in y_batch[:10].numpy()],
        title='Real training faces')

## 2. Build the GAN
The architecture is unchanged from the textbook. Note the two different label inputs.

In [ ]:
# Critic: image (64, 64, 3) + label MAP (64, 64, 2), concatenated as 5 input channels
critic_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
label_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CLASSES))
x = layers.Concatenate(axis=-1)([critic_input, label_input])
x = layers.Conv2D(64, kernel_size=4, strides=2, padding='same')(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2D(128, kernel_size=4, strides=2, padding='same')(x)
x = layers.LeakyReLU()(x)
x = layers.Dropout(0.3)(x)
x = layers.Conv2D(128, kernel_size=4, strides=2, padding='same')(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Dropout(0.3)(x)
x = layers.Conv2D(128, kernel_size=4, strides=2, padding='same')(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Dropout(0.3)(x)
x = layers.Conv2D(1, kernel_size=4, strides=1, padding='valid')(x)
critic_output = layers.Flatten()(x)
critic = models.Model([critic_input, label_input], critic_output, name='critic')
critic.summary()

In [ ]:
# Generator: noise VECTOR (32,) + label VECTOR (2,), concatenated to 34 values
generator_input = layers.Input(shape=(Z_DIM,))
label_input = layers.Input(shape=(CLASSES,))
x = layers.Concatenate(axis=-1)([generator_input, label_input])
x = layers.Reshape((1, 1, Z_DIM + CLASSES))(x)
x = layers.Conv2DTranspose(128, kernel_size=4, strides=1, padding='valid', use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
x = layers.BatchNormalization(momentum=0.9)(x)
x = layers.LeakyReLU(0.2)(x)
generator_output = layers.Conv2DTranspose(CHANNELS, kernel_size=4, strides=2, padding='same', activation='tanh')(x)
generator = models.Model([generator_input, label_input], generator_output, name='generator')
generator.summary()

### Conditional WGAN-GP with a modified `train_step` (Task 2)
Lines marked **CHANGED** differ from the textbook:
1. `label_maps()` turns the one-hot vector `(batch, 2)` into a one-hot map `(batch, 64, 64, 2)` with `tf.broadcast_to` (one call instead of two `tf.repeat` calls), using the real image height and width rather than a global constant.
2. Labels are cast to `float32` so they can be concatenated with the float image tensors.
3. The **generator** always receives the vector form and the **critic** always receives the map form, including inside the gradient penalty.
4. The gradient penalty samples the interpolation weight from a **uniform** distribution on [0, 1], as WGAN-GP specifies, instead of the textbook's normal distribution, so interpolated images lie between a real and a fake image.

In [ ]:
class ConditionalWGAN(models.Model):
    def __init__(self, critic, generator, latent_dim, critic_steps, gp_weight):
        super().__init__()
        self.critic = critic
        self.generator = generator
        self.latent_dim = latent_dim
        self.critic_steps = critic_steps
        self.gp_weight = gp_weight

    def compile(self, c_optimizer, g_optimizer):
        super().compile()   # CHANGED: no run_eagerly=True, so the step is compiled (faster)
        self.c_optimizer = c_optimizer
        self.g_optimizer = g_optimizer
        self.c_wass_loss_metric = metrics.Mean(name='c_wass_loss')
        self.c_gp_metric = metrics.Mean(name='c_gp')
        self.c_loss_metric = metrics.Mean(name='c_loss')
        self.g_loss_metric = metrics.Mean(name='g_loss')

    @property
    def metrics(self):
        return [self.c_loss_metric, self.c_wass_loss_metric, self.c_gp_metric, self.g_loss_metric]

    # CHANGED: one helper converts the label VECTOR into the label MAP the critic expects
    @staticmethod
    def label_maps(one_hot_labels, height, width):
        one_hot_labels = tf.cast(one_hot_labels, tf.float32)                 # (B, 2)
        maps = one_hot_labels[:, None, None, :]                              # (B, 1, 1, 2)
        return tf.broadcast_to(maps, [tf.shape(one_hot_labels)[0], height, width, CLASSES])  # (B, H, W, 2)

    def gradient_penalty(self, batch_size, real_images, fake_images, image_one_hot_labels):
        # CHANGED: uniform alpha in [0, 1] (WGAN-GP), not tf.random.normal
        alpha = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
        interpolated = real_images + alpha * (fake_images - real_images)
        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            pred = self.critic([interpolated, image_one_hot_labels], training=True)   # critic gets the MAP
        grads = gp_tape.gradient(pred, [interpolated])[0]
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]) + 1e-12)
        return tf.reduce_mean((norm - 1.0) ** 2)

    def train_step(self, data):
        real_images, one_hot_labels = data
        one_hot_labels = tf.cast(one_hot_labels, tf.float32)                 # CHANGED: float32 vector for the generator
        batch_size = tf.shape(real_images)[0]
        height, width = tf.shape(real_images)[1], tf.shape(real_images)[2]
        image_one_hot_labels = self.label_maps(one_hot_labels, height, width)   # CHANGED: map for the critic

        # ---- Train the critic CRITIC_STEPS times ----
        for _ in range(self.critic_steps):
            random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
            with tf.GradientTape() as tape:
                fake_images = self.generator([random_latent_vectors, one_hot_labels], training=True)      # VECTOR
                fake_predictions = self.critic([fake_images, image_one_hot_labels], training=True)        # MAP
                real_predictions = self.critic([real_images, image_one_hot_labels], training=True)        # MAP
                c_wass_loss = tf.reduce_mean(fake_predictions) - tf.reduce_mean(real_predictions)
                c_gp = self.gradient_penalty(batch_size, real_images, fake_images, image_one_hot_labels)
                c_loss = c_wass_loss + c_gp * self.gp_weight
            c_gradient = tape.gradient(c_loss, self.critic.trainable_variables)
            self.c_optimizer.apply_gradients(zip(c_gradient, self.critic.trainable_variables))

        # ---- Train the generator once ----
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        with tf.GradientTape() as tape:
            fake_images = self.generator([random_latent_vectors, one_hot_labels], training=True)          # VECTOR
            fake_predictions = self.critic([fake_images, image_one_hot_labels], training=True)            # MAP
            g_loss = -tf.reduce_mean(fake_predictions)
        gen_gradient = tape.gradient(g_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(gen_gradient, self.generator.trainable_variables))

        self.c_loss_metric.update_state(c_loss)
        self.c_wass_loss_metric.update_state(c_wass_loss)
        self.c_gp_metric.update_state(c_gp)
        self.g_loss_metric.update_state(g_loss)
        return {m.name: m.result() for m in self.metrics}


cgan = ConditionalWGAN(critic=critic, generator=generator, latent_dim=Z_DIM,
                       critic_steps=CRITIC_STEPS, gp_weight=GP_WEIGHT)

# Quick shape check of the label formats
_x, _y = next(iter(train))
print('images              ', _x.shape)
print('generator label     ', _y.shape, '(one-hot vector)')
print('critic label        ', ConditionalWGAN.label_maps(_y, IMAGE_SIZE, IMAGE_SIZE).shape, '(one-hot map)')

## 3. Train the GAN

In [ ]:
cgan.compile(
    c_optimizer=optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=ADAM_BETA_1, beta_2=ADAM_BETA_2),
    g_optimizer=optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=ADAM_BETA_1, beta_2=ADAM_BETA_2),
)
if LOAD_MODEL:
    generator.load_weights('./checkpoint/generator.weights.h5')
    critic.load_weights('./checkpoint/critic.weights.h5')

# Fixed noise so every progress image uses the same 10 faces
fixed_z = np.random.RandomState(7).normal(size=(10, Z_DIM)).astype('float32')
LABEL0 = np.repeat([[1, 0]], 10, axis=0).astype('float32')   # not blond
LABEL1 = np.repeat([[0, 1]], 10, axis=0).astype('float32')   # blond


class ImageGenerator(callbacks.Callback):
    """Shows the SAME noise vectors with label 0 and label 1, so only the label differs."""
    def on_epoch_end(self, epoch, logs=None):
        step = epoch + 1
        if step % SHOW_EVERY and step != TRAIN_STEPS:
            return
        imgs0 = self.model.generator([fixed_z, LABEL0], training=False).numpy()
        imgs1 = self.model.generator([fixed_z, LABEL1], training=False).numpy()
        display(imgs0, title=f'Step {step}: label 0 (not blond)', save_to=f'./output/step_{step:04d}_label_0.png')
        display(imgs1, title=f'Step {step}: label 1 (blond)', save_to=f'./output/step_{step:04d}_label_1.png')


class LossLogger(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        step = epoch + 1
        if step % LOG_EVERY == 0 or step == TRAIN_STEPS:
            print(f"step {step:5d}  c_loss {logs['c_loss']:8.3f}  c_wass_loss {logs['c_wass_loss']:8.3f}  "
                  f"c_gp {logs['c_gp']:7.3f}  g_loss {logs['g_loss']:8.3f}")


class SubModelCheckpoint(callbacks.Callback):
    """Saves the generator and critic weights (Keras 3 cannot checkpoint the unbuilt wrapper model)."""
    def on_epoch_end(self, epoch, logs=None):
        step = epoch + 1
        if step % SHOW_EVERY == 0 or step == TRAIN_STEPS:
            self.model.generator.save_weights('./checkpoint/generator.weights.h5')   # Keras 3 requires .weights.h5
            self.model.critic.save_weights('./checkpoint/critic.weights.h5')

model_checkpoint_callback = SubModelCheckpoint()

In [ ]:
# The textbook trains EPOCHS * 100 'epochs' of one batch each (2,000 steps)
history = cgan.fit(
    train.repeat(),
    epochs=TRAIN_STEPS,
    steps_per_epoch=1,
    verbose=0,
    callbacks=[model_checkpoint_callback, ImageGenerator(), LossLogger()],
)

generator.save('./models/generator.keras')   # Keras 3 requires .keras
critic.save('./models/critic.keras')

In [ ]:
# Loss curves
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h['c_wass_loss'], alpha=0.6, label='critic Wasserstein loss')
ax[0].plot(h['g_loss'], alpha=0.6, label='generator loss')
ax[0].set_xlabel('Training step'); ax[0].set_title('Critic and generator loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(h['c_gp'], color='tab:green', alpha=0.6)
ax[1].set_xlabel('Training step'); ax[1].set_title('Gradient penalty'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('./output/losses.png', dpi=100); plt.show()

## 4. Generate faces with each label
The same 10 noise vectors are used for both rows, so any difference between the rows is caused by the label alone.

In [ ]:
z_sample = np.random.normal(size=(10, Z_DIM)).astype('float32')
imgs0 = cgan.generator.predict([z_sample, LABEL0], verbose=0)
imgs1 = cgan.generator.predict([z_sample, LABEL1], verbose=0)
display(imgs0, title='Label 0: not blond', save_to='./output/final_label_0.png')
display(imgs1, title='Label 1: blond', save_to='./output/final_label_1.png')

## 5. How accurate is the conditioning?
To measure accuracy instead of judging it by eye, I train a separate classifier that predicts `Blond_Hair` from **real** CelebA faces, check its accuracy on held-out real faces, and then use it to judge 1,000 generated faces for each label.

In [ ]:
# Blond-hair classifier trained on real faces (separate train / validation split)
clf_train, clf_val = utils.image_dataset_from_directory(
    IMG_DIR, labels=int_labels, image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE,
    validation_split=0.1, subset='both', seed=123, interpolation='bilinear')
clf_train = clf_train.map(lambda x, y: (preprocess(x), y)).prefetch(tf.data.AUTOTUNE)
clf_val = clf_val.map(lambda x, y: (preprocess(x), y)).prefetch(tf.data.AUTOTUNE)

blond_rate = n_blond / len(attributes)
class_weight = {0: 0.5 / (1 - blond_rate), 1: 0.5 / blond_rate}   # blond faces are the minority class

classifier = models.Sequential([
    layers.Input((IMAGE_SIZE, IMAGE_SIZE, CHANNELS)),
    layers.Conv2D(32, 3, strides=2, padding='same', activation='relu'),
    layers.Conv2D(64, 3, strides=2, padding='same', activation='relu'),
    layers.Conv2D(128, 3, strides=2, padding='same', activation='relu'),
    layers.Conv2D(128, 3, strides=2, padding='same', activation='relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid'),
], name='blond_classifier')
classifier.compile('adam', 'binary_crossentropy', metrics=['accuracy'])
CLF_STEPS = int(os.environ.get('CLF_STEPS', 400))
classifier.fit(clf_train.repeat(), epochs=3, steps_per_epoch=CLF_STEPS, class_weight=class_weight, verbose=2)

In [ ]:
# Classifier accuracy on held-out REAL faces
y_true, y_prob = [], []
for xb, yb in clf_val.take(int(os.environ.get('VAL_BATCHES', 80))):
    y_true.append(yb.numpy()); y_prob.append(classifier.predict(xb, verbose=0).ravel())
y_true = np.concatenate(y_true); y_pred = (np.concatenate(y_prob) > 0.5).astype(int)

acc = (y_pred == y_true).mean()
tpr = (y_pred[y_true == 1] == 1).mean() if (y_true == 1).any() else float('nan')
tnr = (y_pred[y_true == 0] == 0).mean() if (y_true == 0).any() else float('nan')
print(f'Real validation faces: {len(y_true):,}')
print(f'Accuracy:              {acc:.1%}')
print(f'Blond recall (TPR):    {tpr:.1%}')
print(f'Not-blond recall (TNR):{tnr:.1%}')
print(f'Balanced accuracy:     {(tpr + tnr) / 2:.1%}')
print(f'(Always guessing "not blond" would score {1 - blond_rate:.1%} accuracy but 50% balanced accuracy.)')

In [ ]:
# Judge 1,000 GENERATED faces per label, using the same noise for both labels
N = 1000
z = np.random.normal(size=(N, Z_DIM)).astype('float32')
gen0 = cgan.generator.predict([z, np.repeat([[1, 0]], N, axis=0).astype('float32')], verbose=0, batch_size=200)
gen1 = cgan.generator.predict([z, np.repeat([[0, 1]], N, axis=0).astype('float32')], verbose=0, batch_size=200)
p0 = classifier.predict(gen0, verbose=0, batch_size=200).ravel()
p1 = classifier.predict(gen1, verbose=0, batch_size=200).ravel()

blond_given_0 = (p0 > 0.5).mean()
blond_given_1 = (p1 > 0.5).mean()
cond_acc = ((p1 > 0.5).sum() + (p0 <= 0.5).sum()) / (2 * N)
flip = ((p1 > 0.5) & (p0 <= 0.5)).mean()

print(f'Generated with label 1 (blond)      judged blond: {blond_given_1:.1%}')
print(f'Generated with label 0 (not blond)  judged blond: {blond_given_0:.1%}')
print(f'Conditioning accuracy (label honoured, both labels): {cond_acc:.1%}')
print(f'Same noise: blond with label 1 AND not blond with label 0: {flip:.1%}')
print(f'Mean blond probability: label 0 = {p0.mean():.3f}, label 1 = {p1.mean():.3f}')

plt.figure(figsize=(7, 4))
plt.hist(p0, bins=30, alpha=0.6, label='label 0 (not blond)')
plt.hist(p1, bins=30, alpha=0.6, label='label 1 (blond)')
plt.xlabel('Classifier probability of blond hair'); plt.ylabel('Generated faces'); plt.legend()
plt.title('How the classifier judges generated faces'); plt.grid(alpha=0.3)
plt.savefig('./output/conditioning_hist.png', dpi=100); plt.show()